In [ ]:
import pandas as pd
import numpy as np

# -----------------------
# INPUT / CONFIG
# -----------------------
month_from = "202412"   # older month (YYYYMM)
month_to   = "202512"   # newer month (YYYYMM)
group_cols = ["ICB plus Code", "BNF Chemical Substance plus Code"]

# -----------------------
# Load & merge / derive
# -----------------------
df_main = pd.read_csv("All Drugs - Latest month vs 12m ago.csv")
df_list = pd.read_csv("List Size by ICB.csv")

# Keep only required columns from list size file
df_list = df_list[["ICB plus Code", "Year Month", "List Size"]]

# Merge on BOTH columns
df_merged = df_main.merge(
    df_list,
    on=["ICB plus Code", "Year Month"],
    how="left"
)

# Derive items per patient column
df_merged["Items per Patient"] = df_merged["Items"] / df_merged["List Size"]

# Derive spend per patient column
df_merged["Spend per Patient"] = df_merged["Actual Cost"] / df_merged["List Size"]

# Derive spend per item column (guard divide-by-zero by keeping NaN)
df_merged["Spend per Item"] = (df_merged["Actual Cost"] / df_merged["Items"]).replace([np.inf, -np.inf], np.nan)

# Save main output
df_merged.to_csv("All Drugs National.csv", index=False)
print("Main merge complete and saved as All Drugs National.csv")

# -----------------------
# Build percent-change table per (ICB, BNF)
# -----------------------

import numpy as np

# Ensure Year Month column is string-like and filter the two months
df_merged["Year Month"] = df_merged["Year Month"].astype(str)
df_compare = df_merged[df_merged["Year Month"].isin([month_from, month_to])].copy()

# Keep only relevant columns
req_cols = group_cols + ["Year Month", "Actual Cost", "List Size", "Items"]
missing = [c for c in req_cols if c not in df_compare.columns]
if missing:
    raise KeyError(f"Missing required columns for percent-change calc: {missing}")

df_compare = df_compare[req_cols]

# Aggregate numerator & denominator per group + month (sum)
agg = (
    df_compare
    .groupby(group_cols + ["Year Month"], as_index=False)
    .agg({"Actual Cost": "sum", "List Size": "sum", "Items": "sum"})
)

# Pivot so months are side-by-side for each (ICB, BNF) pair
pivot = agg.pivot_table(
    index=group_cols,
    columns="Year Month",
    values=["Actual Cost", "List Size", "Items"],
    aggfunc="first"
)

# Flatten columns like "Actual Cost_202412"
pivot.columns = [f"{val}_{col}" for val, col in pivot.columns]
pivot = pivot.reset_index()

# Define expected column names
cost_from_col  = f"Actual Cost_{month_from}"
cost_to_col    = f"Actual Cost_{month_to}"
den_from_col   = f"List Size_{month_from}"
den_to_col     = f"List Size_{month_to}"
items_from_col = f"Items_{month_from}"
items_to_col   = f"Items_{month_to}"

# Ensure columns exist (create with 0 for missing — implies not issued)
for c in [cost_from_col, cost_to_col, den_from_col, den_to_col, items_from_col, items_to_col]:
    if c not in pivot.columns:
        pivot[c] = 0.0

# Coerce numeric and treat missing as zero (missing prescribing => zero)
pivot[[cost_from_col, cost_to_col, den_from_col, den_to_col, items_from_col, items_to_col]] = pivot[
    [cost_from_col, cost_to_col, den_from_col, den_to_col, items_from_col, items_to_col]
].apply(pd.to_numeric, errors="coerce").fillna(0.0)

# Compute rates for each month
# Policy: denom == 0 => rate 0
pivot["spend_per_patient_from"] = np.where(pivot[den_from_col]   == 0, 0.0, pivot[cost_from_col]  / pivot[den_from_col])
pivot["spend_per_patient_to"]   = np.where(pivot[den_to_col]     == 0, 0.0, pivot[cost_to_col]    / pivot[den_to_col])
pivot["items_per_patient_from"] = np.where(pivot[den_from_col]   == 0, 0.0, pivot[items_from_col] / pivot[den_from_col])
pivot["items_per_patient_to"]   = np.where(pivot[den_to_col]     == 0, 0.0, pivot[items_to_col]   / pivot[den_to_col])
pivot["spend_per_item_from"]    = np.where(pivot[items_from_col] == 0, 0.0, pivot[cost_from_col]  / pivot[items_from_col])
pivot["spend_per_item_to"]      = np.where(pivot[items_to_col]   == 0, 0.0, pivot[cost_to_col]    / pivot[items_to_col])


# === Vectorised percent-change helper ===
def pct_change_arrays(f_arr, t_arr):
    """
    Computes numeric percent change between two rate arrays.
    All outputs are numeric floats — no strings.

      - both 0        -> 0.0
      - 0 -> positive -> 999999.0   (new — no prior baseline)
      - positive -> 0 -> -100.0    (discontinued)
      - positive -> positive       -> normal % change, rounded to 1 dp
    """
    f = f_arr.astype(float)
    t = t_arr.astype(float)
    num = np.full(len(f), np.nan, dtype=float)

    mask_both_zero = (f == 0) & (t == 0)
    mask_new       = (f == 0) & (t > 0)
    mask_disappear = (f > 0)  & (t == 0)
    mask_normal    = (f > 0)  & (t > 0)

    num[mask_both_zero] = 0.0
    num[mask_new]       = 999999.0
    num[mask_disappear] = -100.0
    num[mask_normal]    = np.round(
        (t[mask_normal] - f[mask_normal]) / f[mask_normal] * 100.0, 1
    )

    return num


# Apply to each metric
pivot["pct_spend_per_patient"] = pct_change_arrays(
    pivot["spend_per_patient_from"].to_numpy(),
    pivot["spend_per_patient_to"].to_numpy()
)
pivot["pct_items_per_patient"] = pct_change_arrays(
    pivot["items_per_patient_from"].to_numpy(),
    pivot["items_per_patient_to"].to_numpy()
)
pivot["pct_spend_per_item"] = pct_change_arrays(
    pivot["spend_per_item_from"].to_numpy(),
    pivot["spend_per_item_to"].to_numpy()
)

# Round base rate columns for neatness
for col in [
    "spend_per_patient_from", "spend_per_patient_to",
    "items_per_patient_from", "items_per_patient_to",
    "spend_per_item_from",    "spend_per_item_to",
]:
    pivot[col] = pivot[col].round(6)

# Build output dataframe
out_df = pivot[group_cols + [
    "spend_per_patient_from", "spend_per_patient_to", "pct_spend_per_patient",
    "items_per_patient_from", "items_per_patient_to", "pct_items_per_patient",
    "spend_per_item_from",    "spend_per_item_to",    "pct_spend_per_item",
]].rename(columns={
    "spend_per_patient_from": f"Spend per Patient_{month_from}",
    "spend_per_patient_to":   f"Spend per Patient_{month_to}",
    "pct_spend_per_patient":  "% Spend Change",
    "items_per_patient_from": f"Items per Patient_{month_from}",
    "items_per_patient_to":   f"Items per Patient_{month_to}",
    "pct_items_per_patient":  "% Items Change",
    "spend_per_item_from":    f"Spend per Item_{month_from}",
    "spend_per_item_to":      f"Spend per Item_{month_to}",
    "pct_spend_per_item":     "% Spend per Item Change",
})

out_filename = "Spend & Items % Change by BNF and ICB.csv"
out_df.to_csv(out_filename, index=False)

print(f"Saved percent-change table to: {out_filename}")



Main merge complete and saved as All Drugs National.csv
Saved extended percent-change table to: Spend & Items % Change by BNF and ICB.csv
